# 03 — Baseline Classifier: TF-IDF + Logistic Regression

**V1 deliverable.** Train one model that takes a paper abstract and predicts its primary CS category, with confidence scores.

## Pipeline
1. Load cleaned data (output of `01_data_preparation.ipynb`)
2. Encode labels and split train/test (80/20, stratified)
3. TF-IDF vectorization (50k features, unigrams + bigrams)
4. Train logistic regression (multinomial, L2, lbfgs solver)
5. Evaluate on the held-out test set
6. Save model artifacts for `predict.py`

## Configuration choices
- **Features:** TF-IDF with bigrams. `sublinear_tf=True` log-dampens repeated words; `min_df=5` drops typos and rare junk; `max_df=0.95` drops words present in nearly every paper.
- **Model:** Logistic regression with L2 penalty. Empirically the strongest classical model for sparse text classification.
- **Solver:** `lbfgs`. Fast quasi-Newton method, good for L2 multinomial problems.
- **C=5.0:** picked via validation-set tuning on a 30k subsample (see `experiments/` for the tuning notebook).

Multi-label classification, SPECTER2 embeddings, and full hyperparameter search are deferred to V2.

## 1. Imports and configuration

In [1]:
import time
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report

DATA_PATH = Path('../data/processed/arxiv_cs_clean.parquet')
MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(exist_ok=True, parents=True)

SEED = 42

## 2. Load cleaned data

In [2]:
df = pd.read_parquet(DATA_PATH)
print(f'Loaded {len(df):,} papers')
print(f'Categories: {df["first_cat"].nunique()}')
print(f'Year range: {df["year"].min()} — {df["year"].max()}')

Loaded 902,645 papers
Categories: 39
Year range: 2007 — 2026


## 3. Label encoding and train/test split

Stratified split on `first_cat` so each category keeps the same proportion in both train and test. Fixed seed for reproducibility.

In [3]:
le = LabelEncoder()
y = le.fit_transform(df['first_cat'].values)
X_text = df['abstract'].values

X_train_txt, X_test_txt, y_train, y_test = train_test_split(
    X_text, y,
    test_size=0.2,
    stratify=y,
    random_state=SEED,
)
print(f'Train: {len(X_train_txt):,}   Test: {len(X_test_txt):,}')
print(f'Classes: {len(le.classes_)}')

Train: 722,116   Test: 180,529
Classes: 39


## 4. TF-IDF vectorization

Fit on training data only — the test set must be transformed using vocabulary the model has never seen, otherwise the train/test boundary leaks.

In [4]:
t0 = time.time()
vectorizer = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    stop_words='english',
    min_df=5,
    max_df=0.95,
    sublinear_tf=True,
    dtype=np.float32,
)
X_train = vectorizer.fit_transform(X_train_txt)
X_test  = vectorizer.transform(X_test_txt)
vectorize_time = time.time() - t0

print(f'Vectorization: {vectorize_time:.1f}s')
print(f'X_train shape: {X_train.shape}   non-zeros: {X_train.nnz:,}')
print(f'X_test  shape: {X_test.shape}    non-zeros: {X_test.nnz:,}')

Vectorization: 449.4s
X_train shape: (722116, 50000)   non-zeros: 73,872,244
X_test  shape: (180529, 50000)    non-zeros: 18,475,639


## 5. Train logistic regression



In [5]:
t0 = time.time()
clf = LogisticRegression(
    solver='lbfgs',
    penalty='l2',
    C=5.0,
    max_iter=350,
    tol=1e-4,
    n_jobs=-1,
    random_state=SEED,
)
clf.fit(X_train, y_train)
train_time = time.time() - t0

print(f'Training done in {train_time/60:.1f} min')
print(f'Iterations used: {clf.n_iter_[0]}')

Training done in 61.9 min
Iterations used: 350


## 6. Evaluate on the test set

In [6]:
t0 = time.time()
y_pred = clf.predict(X_test)
inference_time = time.time() - t0

accuracy = accuracy_score(y_test, y_pred)
f1_macro = f1_score(y_test, y_pred, average='macro')
f1_weighted = f1_score(y_test, y_pred, average='weighted')

print(f'Inference on {len(y_test):,} papers: {inference_time:.1f}s')
print(f'Accuracy      : {accuracy:.4f}')
print(f'F1 (macro)    : {f1_macro:.4f}')
print(f'F1 (weighted) : {f1_weighted:.4f}')

Inference on 180,529 papers: 1.7s
Accuracy      : 0.7345
F1 (macro)    : 0.6068
F1 (weighted) : 0.7289


In [7]:
print(classification_report(y_test, y_pred, target_names=le.classes_, digits=3, zero_division=0))

              precision    recall  f1-score   support

       cs.AI      0.476     0.362     0.411      9440
       cs.AR      0.635     0.620     0.628      1052
       cs.CC      0.657     0.567     0.608      1715
       cs.CE      0.452     0.308     0.367      1527
       cs.CG      0.725     0.667     0.694      1227
       cs.CL      0.784     0.820     0.802     16327
       cs.CR      0.740     0.745     0.742      6803
       cs.CV      0.859     0.890     0.874     32222
       cs.CY      0.501     0.502     0.502      3174
       cs.DB      0.706     0.646     0.675      1561
       cs.DC      0.628     0.598     0.613      3626
       cs.DL      0.755     0.723     0.739       910
       cs.DM      0.592     0.575     0.583      2006
       cs.DS      0.661     0.677     0.669      3997
       cs.ET      0.533     0.498     0.515       864
       cs.FL      0.637     0.585     0.610       756
       cs.GR      0.567     0.346     0.430       814
       cs.GT      0.766    

## 7. Sanity-check prediction

Pick one paper from the test set and show the top-5 predicted categories with probabilities. This is the same operation `predict.py` performs.

In [8]:
idx = 500
abstract = X_test_txt[idx]
true_label = le.inverse_transform([y_test[idx]])[0]

vec = vectorizer.transform([abstract])
proba = clf.predict_proba(vec)[0]
top5 = np.argsort(proba)[::-1][:5]

print(f'TRUE LABEL: {true_label}\n')
print(f'ABSTRACT (first 300 chars):\n{abstract[:300]}...\n')
print('TOP 5 PREDICTIONS:')
for i in top5:
    marker = '  <-- correct' if le.classes_[i] == true_label else ''
    print(f'  {le.classes_[i]:8s}  {proba[i]:.4f}{marker}')

TRUE LABEL: cs.CV

ABSTRACT (first 300 chars):
While high-quality texture maps are essential for realistic 3D asset rendering, few studies have explored learning directly in the texture space, especially on large-scale datasets. In this work, we depart from the conventional approach of relying on pre-trained 2D diffusion models for test-time opt...

TOP 5 PREDICTIONS:
  cs.CV     0.9724  <-- correct
  cs.GR     0.0232
  cs.LG     0.0031
  cs.AI     0.0005
  cs.RO     0.0002


## 8. Save model artifacts

`predict.py` loads these four files and nothing else.

In [9]:
with open(MODELS_DIR / 'v1_tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)
with open(MODELS_DIR / 'v1_logreg.pkl', 'wb') as f:
    pickle.dump(clf, f)
with open(MODELS_DIR / 'v1_label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

metrics = {
    'model': 'tfidf_bigram+logreg',
    'config': {
        'max_features': 50000,
        'ngram_range': [1, 2],
        'sublinear_tf': True,
        'min_df': 5,
        'max_df': 0.95,
        'C': 5.0,
        'penalty': 'l2',
        'solver': 'lbfgs',
        'max_iter': 350,
        'class_weight': None,
    },
    'n_train': int(len(y_train)),
    'n_test': int(len(y_test)),
    'n_classes': int(len(le.classes_)),
    'vocab_size': int(X_train.shape[1]),
    'train_time_sec': float(train_time),
    'n_iter': int(clf.n_iter_[0]),
    'accuracy': float(accuracy),
    'f1_macro': float(f1_macro),
    'f1_weighted': float(f1_weighted),
    'sklearn_version': sklearn.__version__,
}

with open(MODELS_DIR / 'v1_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print(f'Saved 4 files to {MODELS_DIR.resolve()}:')
for name in ['v1_tfidf_vectorizer.pkl', 'v1_logreg.pkl', 'v1_label_encoder.pkl', 'v1_metrics.json']:
    path = MODELS_DIR / name
    size_mb = path.stat().st_size / 1e6
    print(f'  {name}  ({size_mb:.2f} MB)')

Saved 4 files to C:\Users\Moham\PycharmProjects\arxiv-intelligence-search\models:
  v1_tfidf_vectorizer.pkl  (1.95 MB)
  v1_logreg.pkl  (15.60 MB)
  v1_label_encoder.pkl  (0.00 MB)
  v1_metrics.json  (0.00 MB)
